## PART 2: The Root User

**Support context:** Nearly every serious support incident touches privilege at some point — "permission denied," "who has access to this," "why did this script fail silently." Understanding root and _why we avoid using it directly_ is foundational to reading almost any incident correctly.

> **Concept:** Root is not "an admin" the way Windows has admin accounts. Root is **UID 0** — a specific numeric user ID that the Linux kernel hardcodes to **bypass essentially all permission checks**. Any user account with UID 0 _is_ root, regardless of username. Let's confirm this live.

In [ ]:
dennie@HP:~/linux-tut$ id
uid=1000(dennie) gid=1000(dennie) groups=1000(dennie),4(adm),24(cdrom),27(sudo),30(dip),46(plugdev),100(users)

In [ ]:
dennie@HP:~/linux-tut$ grep root /etc/passwd
root:x:0:0:root:/root:/bin/bash

In [ ]:
dennie@HP:~/linux-tut$ id root
uid=0(root) gid=0(root) groups=0(root)

**READ THE OUTPUT:**

- Your `id` shows `uid=1000` — a normal user. Note `groups=...,27(sudo),...` — you're already a member of the **`sudo` group**, which is _why_ `sudo` works for your account at all. That's not automatic for every user.
- `/etc/passwd`'s root line: field 3 (`0`) is the UID, field 4 (`0`) is the GID. **UID 0 is the actual definition of "being root."** Not the username `root` — the number `0`. You could rename the `root` account to `admin` and it would still have full power, because the kernel checks the number, not the name.
- `id root` confirms it: `uid=0(root) gid=0(root)`.

**Developer bridge:** think of UID 0 like a Django `is_superuser=True` flag baked into the OS kernel itself — except instead of your app code checking that flag before every sensitive action, the _kernel_ checks it before every single syscall (open a file, kill a process, bind to a low port, etc.). There's no "route" a superuser process can't reach.

**GOTCHA:** People say "log in as root" meaning "become UID 0." On this WSL box, root's login shell (`/bin/bash`) and home (`/root`) are visible in `/etc/passwd` exactly like any other user — root is just a _very privileged_ regular-looking account entry.

### Why Not Root, and How sudo Delegates Safely
**Why never run as root day-to-day:**

1. **No safety net.** As root, `rm -rf /some/typo/path` just runs — the kernel doesn't ask **"are you sure,"** because root is _defined_ as the account **nothing gets asked of**. As a normal user, a permissions error is often the only thing standing between a typo and a wrecked system.
2. **No audit trail.** If every admin on a team just does `su -` and works as `root`, your logs show `root did X` for _everyone_ — you can't tell which human actually did it. With `sudo`, every escalated command is logged against the real username.
3. **Blast radius of compromise.** If a process you're running (a script, a compromised dependency, a malicious download) misbehaves while you're root, it inherits root's full power instantly. As a normal user, it's limited to what _you_ can touch — which is the whole idea behind the **principle of least privilege**: give an account only the access it needs for right now, not everything it might ever need.

**Developer bridge:** this is identical logic to why you don't run your Django app as the Postgres superuser — you create a scoped DB role with just the grants that app needs. Root vs. a sudo-enabled user is the OS-level version of that same instinct.

#### `sudo` vs `su` — different mechanisms, not just different commands

- **`su - username`**: starts a **new login shell** _as_ that user. `su -` with no name means "become root, fully" — you're now root until you `exit`. Requires **root's own password** (on distros where root has one set).
- **`sudo command`**: runs **_one command_ with elevated privileges**, then you're immediately back to being yourself. Requires **your own password** (proving it's really you), and only works if you're explicitly authorized in `/etc/sudoers`.

Here's the mechanism that makes `sudo` (and `su`) possible at all — the setuid bit:

In [ ]:
dennie@HP:~/linux-tut$ ls -l "$(which sudo)" "$(which su)"
-rwsr-xr-x 1 root root  55680 Mar  6 16:00 /usr/bin/su
-rwsr-xr-x 1 root root 277936 Mar  2 12:56 /usr/bin/sudo

**READ THE OUTPUT:** see that lowercase `s` where the owner's execute bit normally goes (`rws` instead of `rwx`)? That's **setuid** — a permission bit meaning "when _anyone_ runs this binary, it executes as if the file's _owner_ (root) ran it," not as the invoking user. That's the entire trick: `sudo` and `su` are owned by root and marked setuid, so they can briefly act with root's power on your behalf, check your credentials, then hand you exactly what `/etc/sudoers` says you're allowed. (This is Pillar 3 territory — we'll cover setuid/setgid/sticky bit properly later, but you can't understand _how_ sudo works without seeing it once.)

---

**Recap of what's now solid:**

- Root = UID 0, not a username — kernel checks the number
- Never work as root directly: no safety net, no per-person audit trail, full blast radius if compromised
- `sudo`/`su` work via the **setuid bit** (`rws` on the binary) — briefly borrow root's power, then hand it back
- `/etc/sudoers` (mode `440`, root-only) defines who can escalate; your access comes from being in the `sudo` **group**, not a special sudoers entry for you personally
- `sudo` logs _who_ did _what_ as _whom_ (`/var/log/auth.log`); `su -` doesn't give you that
- `sudo command` > `sudo -s` > `sudo -i` > `su -` in terms of footprint — default to the smallest one that gets the job done
- On this box, root's password is locked — `su -` is a dead end by design